# 서울교통라이브 데이터 전처리와 분석

서울교통라이브 `feature/newsroom-integration`의 [검증 기준 커밋 6c395d1](https://github.com/schwatara-eng/team1_traffic_live/tree/6c395d158508f99625a0a3d203f15bf1b67b63a7/data)을 기준으로 작성했어. VS Code에서 프로젝트 폴더를 열고 이 노트북을 프로젝트 루트에 둔 뒤 위에서 아래로 실행해. Python 환경에 pandas와 Jupyter 커널(ipykernel)이 필요해.

전체 CSV의 상태를 살펴보고 서울 고령 보행사고, 보호구역, OECD 순위를 재계산해. 원자료는 읽기만 하고, 출처는 저장소 파일 단위로 표시해. 공식 원문 대조는 기사 발행 전에 별도로 해야 해.

In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# 프로젝트 루트에서 실행해. 다른 위치라면 이 경로만 바꿔.
DATA_DIR = Path("data")
if not DATA_DIR.is_dir():
    raise FileNotFoundError("프로젝트 루트에서 실행하거나 DATA_DIR을 수정해.")

files = sorted(DATA_DIR.rglob("*.csv"))
if not files:
    raise FileNotFoundError(f"CSV 파일이 없어. 경로를 확인해. {DATA_DIR.resolve()}")

tables = {
    path.relative_to(DATA_DIR).as_posix(): pd.read_csv(
        path, encoding="utf-8-sig", dtype="string"
    ).apply(lambda column: column.str.strip()).replace("", pd.NA)
    for path in files
}

inventory = pd.DataFrame([
    {
        "파일": name,
        "행": len(table),
        "열": len(table.columns),
        "전체 행 중복": int(table.duplicated().sum()),
        "결측 셀": int(table.isna().sum().sum()),
    }
    for name, table in tables.items()
])
display(inventory)
print(f"CSV {len(tables)}개 로드 완료")


,파일,행,열,전체 행 중복,결측 셀
0,01_자치구연도_패널.csv,100,21,0,150
1,02_사고상세_동대문도봉.csv,764,30,0,0
2,03_다발지역_서울.csv,1554,20,0,0
3,04_보호구역_마스터.csv,197,17,0,331
4,05_보호구역_경계.csv,280,20,0,4
5,06_다발지역_보호구역_연결.csv,1554,17,0,0
6,07_다발지역_보호구역_500m쌍.csv,1409,7,0,0
7,09_어린이보호구역_서울_지도용.csv,1601,18,0,4262
8,10_어린이보호구역_교통안전_요약.csv,8,10,0,21
9,11_서울_연령별_주민등록인구_202609.csv,101,6,0,0


CSV 23개 로드 완료


## 서울 고령 보행사고

자치구·연도 키가 유일하고 연도마다 25개 구가 있는지 확인해. 핵심 수치는 숫자로 변환하고 결측이나 음수가 있으면 집계를 중단해. 자치구별 비율의 단순 평균 대신 서울 전체 사고 건수를 서울 전체 고령인구로 나눠.

In [2]:
panel = tables["01_자치구연도_패널.csv"].copy()
keys = ["사고연도", "자치구코드"]
assert not panel[keys].isna().any().any(), "자치구·연도 키에 결측이 있어."
assert not panel.duplicated(keys).any(), "자치구·연도 키가 중복돼."
assert panel.groupby("사고연도")["자치구코드"].nunique().eq(25).all(), (
    "25개 구가 모두 들어 있는지 확인해."
)

columns = {
    "고령보행자_사고건수": "사고건수",
    "고령보행자_사망자수": "사망자수",
    "고령보행자_부상자수": "부상자수",
    "고령인구": "고령인구",
}
numeric = panel[list(columns)].apply(pd.to_numeric, errors="raise")
assert numeric.notna().all().all(), "합산 대상에 결측이 있어."
assert numeric.ge(0).all().all(), "합산 대상에 음수가 있어."
assert numeric.mod(1).eq(0).all().all(), "건수 또는 인구에 소수가 들어 있어."
annual = (
    numeric.astype("int64")
    .rename(columns=columns)
    .groupby(panel["사고연도"])
    .sum()
    .sort_index()
)
assert annual["고령인구"].gt(0).all(), "사고율 분모를 확인해."
annual["고령인구 1만 명당 사고"] = (
    annual["사고건수"] / annual["고령인구"] * 10_000
).round(2)
annual["전년 대비 사고건수"] = annual["사고건수"].diff().astype("Int64")
annual["전년 대비 사망자수"] = annual["사망자수"].diff().astype("Int64")
display(annual)

living_coverage = (
    panel.groupby("사고연도")["고령생활인구_연평균"].count()
    .rename("고령생활인구 값이 있는 구")
)
display(living_coverage)


,사고건수,사망자수,부상자수,고령인구,고령인구 1만 명당 사고,전년 대비 사고건수,전년 대비 사망자수
사고연도,,,,,,,
2022,1846,65,1798,1667411,11.07,<NA>,<NA>
2023,1973,44,1957,1743696,11.32,127,-21
2024,1963,68,1921,1824895,10.76,-10,24
2025,2060,82,2010,1912751,10.77,97,14


사고연도
2022     0
2023     0
2024     0
2025    25
Name: 고령생활인구 값이 있는 구, dtype: int64

## 보호구역 기록 집계

노인보호구역은 2025년 명단에 포함된 시설을 세고, 어린이보호구역은 09번 파일의 시설명과 도로명주소 조합을 점검해. 경찰청 12번의 서울 집계는 기준이 다를 수 있어 별도 행에 표시해.

In [ ]:
elder_zones = tables["04_보호구역_마스터.csv"]
child_zones = tables["09_어린이보호구역_서울_지도용.csv"]
police = tables["12_경찰청_어린이보호구역_지정현황_20260630.csv"]
police_seoul = police.loc[police["지역"].eq("서울")]
assert len(police_seoul) == 1, "경찰청 표의 서울 행을 확인해."
assert elder_zones["시설ID"].notna().all(), "노인 시설ID에 결측이 있어."
assert elder_zones["시설ID"].is_unique, "노인 시설ID가 중복돼."
assert not child_zones.duplicated(["시설명", "도로명주소"]).any(), (
    "어린이 시설명·주소 중복을 확인해."
)

zone_counts = pd.DataFrame([
    ("노인보호구역 명단", "2025년 명단 포함 시설", int(elder_zones["명단2025"].eq("1").sum())),
    ("어린이보호구역 09번", "서울 시설 기록", len(child_zones)),
    ("어린이보호구역 경찰청", "2026-06-30 서울 기재항목 합계", int(police_seoul["기재항목합계"].iloc[0])),
], columns=["자료", "기준", "수"])
display(zone_counts)
print("09번 지정여부 결측:", int(child_zones["지정여부"].isna().sum()))


,자료,기준,수
0,노인보호구역 명단,2025년 명단 포함 시설,195
1,어린이보호구역 09번,서울 시설 기록,1601
2,어린이보호구역 경찰청,2026-06-30 서울 기재항목 합계,1630


09번 지정여부 결측: 1601


## OECD 한국 순위 재계산

회원국 중 해당 연도 지표가 있는 국가만 비교하고, 값이 같으면 공동 순위로 처리해. 전체 교통사고 사망률과 보행 중 사망률은 둘 다 65세 이상 인구 10만 명당이지만 사망자 범위가 달라.

In [4]:
oecd = tables["13_OECD_고령지표_wide.csv"].copy()
oecd = oecd.loc[
    oecd["국가구분"].eq("회원국") & oecd["연도"].isin(["2020", "2021", "2022", "2023"])
].copy()
assert not oecd.duplicated(["ISO3", "연도"]).any(), "국가·연도 중복이 있어."

metrics = {
    "전체 교통사고": "65세이상_인구10만명당_사망자수",
    "보행 중": "65세이상_인구10만명당_보행중_사망자수",
}
korea_results = []
for label, column in metrics.items():
    oecd[column] = pd.to_numeric(oecd[column], errors="raise")
    by_year = oecd.groupby("연도")[column]
    korea_results.append(oecd.assign(
        지표=label,
        사망률=oecd[column],
        순위=by_year.rank(ascending=False, method="min").astype("Int64"),
        비교국수=by_year.transform("count"),
    ).loc[oecd["ISO3"].eq("KOR"), ["연도", "지표", "사망률", "순위", "비교국수"]])

korea = pd.concat(korea_results).sort_values(["연도", "지표"])
display(korea)


,연도,지표,사망률,순위,비교국수
43,2020,보행 중,7.7,1,29
43,2020,전체 교통사고,16.5,1,29
82,2021,보행 중,7.0,2,31
82,2021,전체 교통사고,15.2,2,32
121,2022,보행 중,6.2,2,32
121,2022,전체 교통사고,13.9,2,32
160,2023,보행 중,5.8,2,32
160,2023,전체 교통사고,13.1,5,35


## 기사에 사용할 수치 확인

기준 커밋의 정상 결과는 **2024년 1,963건 → 2025년 2,060건, 97건 증가**, **고령 보행 사망 68명 → 82명, 14명 증가**야. OECD 표에서 2023년 한국은 **전체 35개국 중 5위**, **보행 중 32개국 중 2위**로 나와. 데이터가 업데이트되면 계산 결과도 달라질 수 있어.

In [5]:
facts = pd.DataFrame([
    ("서울 고령 보행자 사고", "서울", "2025", annual.loc["2025", "사고건수"], "건", "01_자치구연도_패널.csv"),
    ("서울 고령 보행자 사고 전년 대비 증가", "서울", "2024→2025", annual.loc["2025", "전년 대비 사고건수"], "건", "01_자치구연도_패널.csv"),
    ("서울 고령 보행자 사망", "서울", "2025", annual.loc["2025", "사망자수"], "명", "01_자치구연도_패널.csv"),
], columns=["지표", "지역", "기간", "값", "단위", "출처_파일"])
display(facts)

# 팩트시트 CSV가 필요할 때 아래 줄의 주석을 풀고 실행해.
# facts.to_csv("traffic_facts.csv", index=False, encoding="utf-8-sig")


,지표,지역,기간,값,단위,출처_파일
0,서울 고령 보행자 사고,서울,2025,2060,건,01_자치구연도_패널.csv
1,서울 고령 보행자 사고 전년 대비 증가,서울,2024→2025,97,건,01_자치구연도_패널.csv
2,서울 고령 보행자 사망,서울,2025,82,명,01_자치구연도_패널.csv


In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()  # 프로젝트 루트. 아니면 Path(r"C:\workspace_3\team1_traffic_live")
YEAR = 2025

panel = pd.read_csv(ROOT / "data" / "01_자치구연도_패널.csv", encoding="utf-8-sig")

def by_year(year):
    d = panel[panel["사고연도"] == year].set_index("자치구명")
    assert len(d) == 25, f"{year}년 자치구가 25개가 아니야: {len(d)}"
    return d

cur, prev = by_year(YEAR), by_year(YEAR - 1).loc[by_year(YEAR).index]

rate = cur["고령보행자_사고건수"] / cur["고령인구"] * 10000
prev_rate = prev["고령보행자_사고건수"] / prev["고령인구"] * 10000
seoul_rate = cur["고령보행자_사고건수"].sum() / cur["고령인구"].sum() * 10000

rank = rate.rank(ascending=False, method="min").astype(int)
prev_rank = prev_rate.rank(ascending=False, method="min").astype(int)
change = cur["고령보행자_사고건수"] - prev["고령보행자_사고건수"]

table = pd.DataFrame({
    "자치구": cur.index,
    "순위": rank.values,
    "전년 순위": prev_rank.values,
    "순위 변화(상승 +)": (prev_rank - rank).values,
    "사고율": rate.round(2).values,
    "서울 대비 격차": (rate - seoul_rate).round(2).values,
    "사고 증감(건)": change.values,
    "사고 증감률(%)": (change / prev["고령보행자_사고건수"] * 100).round(1).values,
}).sort_values(["순위", "자치구"]).reset_index(drop=True)

print(f"서울 전체 1만 명당 사고율: {seoul_rate:.2f}")
table

서울 전체 1만 명당 사고율: 10.77


,자치구,순위,전년 순위,순위 변화(상승 +),사고율,서울 대비 격차,사고 증감(건),사고 증감률(%)
0,중구,1,1,0,22.08,11.31,0.0,0.0
1,동대문구,2,3,1,18.14,7.37,2.0,1.5
2,종로구,3,2,-1,16.47,5.70,-9.0,-15.0
3,금천구,4,10,6,16.38,5.61,29.0,53.7
4,영등포구,5,11,6,14.43,3.66,27.0,34.2
5,강북구,6,4,-2,14.11,3.34,11.0,11.7
6,중랑구,7,15,8,11.91,1.14,17.0,19.3
7,강남구,8,6,-2,11.83,1.06,3.0,2.7
8,강동구,9,13,4,11.51,0.74,13.0,13.0
9,은평구,10,9,-1,11.42,0.65,6.0,5.4
